# **Cross-Modal Retrieval**

**Cross-modal retrieval** means querying with one modality to find matching items in another — most commonly **text→image** ("find images matching this caption") and **image→text** ("find the caption for this image"). It is the practical engine behind image search, product search, and dataset curation, and it is the most direct application of a **shared embedding space**.

## **1. The shared embedding space**

Retrieval works when images and texts are mapped into the *same* vector space such that a matching pair has high similarity. Models like **CLIP** produce L2-normalized embeddings where similarity is the **cosine** (equivalently a dot product after normalization). Given a query, we rank candidates by similarity and return the top-$k$.

Because both modalities share one space, all of an item's neighbors — in *either* modality — are simply its nearest vectors.

## **2. Contrastive learning trains the space**

The space is learned with a contrastive objective (**InfoNCE**, see the index notebook): matched image-text pairs are pulled together, mismatched pairs pushed apart. After training:

- embeddings are **normalized**, so cosine similarity $\in [-1, 1]$;
- a learned **temperature** scales logits during training but is irrelevant at retrieval time (it doesn't change the ranking).

The quality of retrieval is therefore inherited directly from the contrastive pretraining.

## **3. Evaluating retrieval: Recall@K**

The standard metric is **Recall@K (R@K)**: the fraction of queries whose correct match appears in the top-$K$ retrieved results. Reported as **R@1 / R@5 / R@10** for both directions (image→text and text→image), typically on Flickr30k or MS-COCO.

- **R@1** — strict: the true match must rank first.
- **R@5 / R@10** — more forgiving; useful when several candidates are plausible.

A related summary metric is **median rank** of the correct item. Higher R@K and lower median rank are better.

## **4. Scaling with approximate nearest neighbors (ANN)**

Exact retrieval scores the query against *every* candidate — $O(N)$ per query, fine for thousands of items but too slow for millions. **ANN** indexes (e.g. **FAISS**, HNSW, IVF-PQ) trade a tiny bit of accuracy for sub-linear search. Typical workflow:

1. Precompute and **normalize** all candidate embeddings once.
2. Build an index (inner-product / cosine metric).
3. At query time, embed the query and ask the index for the top-$k$ neighbors.

Because embeddings are precomputed, adding new items only requires encoding and inserting them — no retraining.

## **5. Code: CLIP image↔text similarity**

```python
from transformers import CLIPModel, CLIPProcessor
from PIL import Image
import torch
import torch.nn.functional as F

model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

images = [Image.open(p).convert("RGB") for p in ["a.jpg", "b.jpg", "c.jpg"]]
texts = ["a dog on a beach", "a plate of sushi", "a city skyline at night"]

inputs = processor(text=texts, images=images, return_tensors="pt", padding=True)
with torch.no_grad():
    img_emb = F.normalize(model.get_image_features(pixel_values=inputs["pixel_values"]), dim=-1)
    txt_emb = F.normalize(model.get_text_features(
        input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"]), dim=-1)

# cosine similarity matrix: rows = images, cols = texts
sim = img_emb @ txt_emb.t()          # (num_images, num_texts)

# text -> image retrieval: best image for each caption
best_image = sim.argmax(dim=0)       # index into `images` per text
for t, idx in zip(texts, best_image):
    print(f"{t!r}  ->  image #{int(idx)}")
```

For large corpora, replace the dense `img_emb @ txt_emb.t()` with a FAISS index built over the precomputed embeddings.

**References**
- Radford et al., *CLIP*, 2021.
- Johnson et al., *Billion-scale similarity search with GPUs* (FAISS), 2017.
- Plummer et al., *Flickr30k Entities*, 2015.